In [1]:
import os
import requests
import browser_cookie3
from dotenv import load_dotenv

load_dotenv()

LEAGUE_ID = 1039832288
SEASON = 2026

URL = (
    f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/wfba/"
    f"seasons/{SEASON}/segments/0/leagues/{LEAGUE_ID}"
)

PARAMS = [
    ("view", "mTeam"),
    ("view", "mRoster"),
    ("view", "mSettings"),
    ("view", "mMatchup"),
    ("view", "mScoreboard"),
]

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json,text/plain,*/*",
    "Referer": f"https://fantasy.espn.com/womens-basketball/league?leagueId={LEAGUE_ID}",
    "Origin": "https://fantasy.espn.com",
}


def get_espn_cookies():
    """
    First try .env cookies.
    If missing/expired, pull fresh cookies from your logged-in browser.
    """
    espn_s2 = os.getenv("ESPN_S2")
    swid = os.getenv("SWID")

    if espn_s2 and swid:
        return {"espn_s2": espn_s2, "SWID": swid}

    cj = browser_cookie3.chrome(domain_name=".espn.com")
    cookies = {c.name: c.value for c in cj}

    if "espn_s2" not in cookies or "SWID" not in cookies:
        raise RuntimeError(
            "Could not find ESPN cookies. Log into ESPN Fantasy in Chrome first."
        )

    return {
        "espn_s2": cookies["espn_s2"],
        "SWID": cookies["SWID"],
    }


def fetch_league_data():
    session = requests.Session()
    session.cookies.update(get_espn_cookies())

    r = session.get(
        URL,
        params=PARAMS,
        headers=HEADERS,
        allow_redirects=False,
        timeout=20,
    )

    print("status:", r.status_code)
    print("content-type:", r.headers.get("content-type"))
    print("location:", r.headers.get("location"))

    if r.status_code in (301, 302, 303, 307, 308):
        raise RuntimeError("Redirected by ESPN. Cookies are invalid/expired.")

    if "application/json" not in r.headers.get("content-type", ""):
        raise RuntimeError(f"ESPN did not return JSON:\n{r.text[:500]}")

    return r.json()


data = fetch_league_data()


status: 200
content-type: application/json;charset=utf-8
location: None


In [2]:
import json
from datetime import datetime
from zoneinfo import ZoneInfo
from cbb.lib import paths

ET = ZoneInfo("America/New_York")

pulled_at = datetime.now(ET)

output = {
    "metadata": {
        "pulled_at": pulled_at.isoformat(),
        "pulled_at_readable": pulled_at.strftime("%Y-%m-%d %I:%M:%S %p %Z"),
        "league_id": LEAGUE_ID,
        "season": SEASON,
    },
    "data": data,
}

out_path = paths.DATA / "wnba.json"
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved fresh data pull: {output['metadata']['pulled_at_readable']}")

Saved fresh data pull: 2026-05-17 01:08:04 PM EDT


In [3]:
from copy import deepcopy

def organize_by_team(data):
    league_data = data["data"] if "data" in data else data

    teams_by_id = {}

    for raw_team in league_data.get("teams", []):
        team = deepcopy(raw_team)  # keeps ALL original ESPN data

        team_id = team.get("id")

        # Add helper fields without deleting anything
        team["_parsed"] = {
            "team_id": team_id,
            "abbrev": team.get("abbrev"),
            "display_name": (
                team.get("name")
                or " ".join(
                    part for part in [
                        team.get("location"),
                        team.get("nickname")
                    ]
                    if part
                )
                or team.get("abbrev")
                or f"Team {team_id}"
            ),
            "roster_count": len(team.get("roster", {}).get("entries", [])),
        }

        teams_by_id[team_id] = team

    return teams_by_id

In [4]:
teams = organize_by_team(data)

with open("team_dump.json", "w") as f:
    json.dump(teams, f, indent=2)

print("Saved full team dump to team_dump.json")

Saved full team dump to team_dump.json


In [34]:
MINUTES_STAT_ID = "40"

STARTER_SLOTS = {0, 1, 2, 3, 4, 5}
BENCH_SLOTS = {6, 7, 8, 9}

MAX_MINUTES_PER_GAME = 40


def get_player_minutes_by_day(entry):
    player = entry["playerPoolEntry"]["player"]
    stat_rows = player.get("stats", [])

    minutes_by_day = {}

    for row in stat_rows:
        scoring_period = row.get("scoringPeriodId")

        # scoringPeriodId 0 is usually ESPN's matchup/aggregate row
        # we only want real daily scoring periods
        if not scoring_period:
            continue

        minutes = float(row.get("stats", {}).get(MINUTES_STAT_ID, 0.0))
        minutes_by_day[scoring_period] = minutes_by_day.get(scoring_period, 0.0) + minutes

    return minutes_by_day


def parse_matchup_team(team_data, team_lookup):
    team_id = team_data["teamId"]
    team_info = team_lookup.get(team_id, {})

    entries = team_data.get("rosterForMatchupPeriod", {}).get("entries", [])

    players = []
    team_minutes_by_day = {}

    for entry in entries:
        lineup_slot = entry.get("lineupSlotId")
        is_starter = lineup_slot in STARTER_SLOTS

        player = entry["playerPoolEntry"]["player"]
        minutes_by_day = get_player_minutes_by_day(entry)

        total_minutes_played = sum(minutes_by_day.values())
        games_played = len([m for m in minutes_by_day.values() if m > 0])

        if is_starter:
            for day, minutes in minutes_by_day.items():
                team_minutes_by_day[day] = team_minutes_by_day.get(day, 0.0) + minutes

        players.append({
            "player_id": player.get("id"),
            "name": player.get("fullName"),
            "lineup_slot_id": lineup_slot,
            "is_starter": is_starter,
            "pro_team_id": player.get("proTeamId"),
            "injury_status": player.get("injuryStatus"),
            "minutes_by_day": minutes_by_day,
            "games_played": games_played,
            "minutes_played": total_minutes_played,
            "max_minutes_used_so_far": games_played * MAX_MINUTES_PER_GAME,
            "max_minutes_left_from_games_played": max(
                0,
                games_played * MAX_MINUTES_PER_GAME - total_minutes_played
            ),
        })

    starter_minutes_played = sum(
        p["minutes_played"]
        for p in players
        if p["is_starter"]
    )

    starter_max_minutes_used_so_far = sum(
        p["max_minutes_used_so_far"]
        for p in players
        if p["is_starter"]
    )

    return {
        "team_id": team_id,
        "team_name": team_info.get("name", f"Team {team_id}"),
        "team_abbrev": team_info.get("abbrev"),
        "total_points": team_data.get("totalPoints"),
        "minutes_by_day": dict(sorted(team_minutes_by_day.items())),
        "minutes_played": starter_minutes_played,
        "max_minutes_used_so_far": starter_max_minutes_used_so_far,
        "max_minutes_left_from_games_played": max(
            0,
            starter_max_minutes_used_so_far - starter_minutes_played
        ),
        "players": players,
    }
def get_team_lookup(league_data):
    teams = {}

    for team in league_data["teams"]:
        team_id = team["id"]

        teams[team_id] = {
            "team_id": team_id,
            "abbrev": team.get("abbrev"),
            "name": (
                team.get("name")
                or team.get("abbrev")
                or f"Team {team_id}"
            )
        }

    return teams


def parse_matchups(league_data):
    team_lookup = get_team_lookup(league_data)

    matchups = []

    for matchup in league_data["schedule"]:

        # skip empty schedule entries
        if "home" not in matchup or "away" not in matchup:
            continue

        parsed = {
            "matchup_id": matchup.get("id"),
            "matchup_period": matchup.get("matchupPeriodId"),
            "home": parse_matchup_team(
                matchup["home"],
                team_lookup
            ),
            "away": parse_matchup_team(
                matchup["away"],
                team_lookup
            )
        }

        matchups.append(parsed)

    return matchups

In [33]:
with open(out_path) as f:
    saved = json.load(f)

league_data = saved["data"]
matchups = parse_matchups(league_data)

for matchup in matchups:
    print("\n" + "=" * 90)
    print(f"MATCHUP {matchup['matchup_id']} | MATCHUP PERIOD {matchup['matchup_period']}")
    print("=" * 90)

    for side in ["away", "home"]:
        team = matchup[side]

        print()
        print(f"{side.upper()}: {team['team_name']} ({team['team_abbrev']})")
        print(f"Points: {team['total_points']}")
        print(f"Starter minutes played: {team['minutes_played']:.1f}")
        print(f"Max starter minutes from games already played: {team['max_minutes_used_so_far']:.1f}")
        print(f"Unused minutes from games already played: {team['max_minutes_left_from_games_played']:.1f}")

        print("\nMinutes by scoring period:")
        for day, minutes in team["minutes_by_day"].items():
            print(f"  Period {day}: {minutes:.1f}")

        print("\nPlayers:")
        for p in team["players"]:
            if not p["is_starter"]:
                continue

            day_text = ", ".join(
                f"P{day}: {mins:.1f}"
                for day, mins in sorted(p["minutes_by_day"].items())
            )

            print(
                f"  {p['name']:<25} "
                f"Total: {p['minutes_played']:>5.1f} | "
                f"Games: {p['games_played']} | "
                f"Unused: {p['max_minutes_left_from_games_played']:>5.1f} | "
                f"{day_text}"
            )


MATCHUP 1 | MATCHUP PERIOD 1

AWAY: Stud bud enthusiast (KST)
Points: 650.0
Starter minutes played: 0.0
Max starter minutes from games already played: 0.0
Unused minutes from games already played: 0.0

Minutes by scoring period:

Players:
  Jacy Sheldon              Total:   0.0 | Games: 0 | Unused:   0.0 | 
  Cheyenne Parker-Tyus      Total:   0.0 | Games: 0 | Unused:   0.0 | 
  Lauren Betts              Total:   0.0 | Games: 0 | Unused:   0.0 | 
  Gabby Williams            Total:   0.0 | Games: 0 | Unused:   0.0 | 
  Caitlin Clark             Total:   0.0 | Games: 0 | Unused:   0.0 | 
  Allisha Gray              Total:   0.0 | Games: 0 | Unused:   0.0 | 
  Betnijah Laney-Hamilton   Total:   0.0 | Games: 0 | Unused:   0.0 | 
  Jessica Shepard           Total:   0.0 | Games: 0 | Unused:   0.0 | 
  Courtney Williams         Total:   0.0 | Games: 0 | Unused:   0.0 | 

HOME: Quarterzip Memorial Team (QMT)
Points: 592.0
Starter minutes played: 0.0
Max starter minutes from games already pl